# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aneeqahabib/FlyRank_ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a **classification** task. The decision is whether a content item should receive attention because its observed performance is declining. Classification fits because each page can receive an observed yes/no label: declining or not declining. The output supports editorial and SEO teams in prioritizing pages for investigation; it does not predict or control search-engine rankings.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Classification means each content item receives a yes/no outcome label.
task_type = "classification"
print("Task type:", task_type)
print("Decision supported: prioritize content items for review.")

Task type: classification
Decision supported: prioritize content items for review.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is `is_declining_label`, defined as `1` when the observed `trend_direction` is `"down"` and `0` otherwise. This label comes from the measured change in impressions between the last 30 days and the previous 30 days.

The model must not use `trend_direction` or `trend_pct` as features because they directly define the label. The target is an observed outcome for this dataset, not a subjective editorial rule. Later validation must also respect the time order between features and the outcome window.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)

print("Target counts:")
print(df["is_declining_label"].value_counts().sort_index())
print("\nTarget rate:", round(df["is_declining_label"].mean(), 3))

Target counts:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Target rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary metric will be **ROC-AUC** on a held-out validation set. It measures whether the model tends to assign higher decline scores to actually declining pages than to non-declining pages across different thresholds.

A score of `0.50` means no better than random ordering. To ground this concretely rather than just asserting it, I computed a constant-probability baseline — predicting the base rate (`0.542`) for every row — which produced **ROC-AUC = `0.500` exactly**, confirming an uninformative model sits at the 0.50 floor regardless of class balance.

Given the base rate of `0.542` and the noisiness of real SEO/content signals, I will treat ROC-AUC ≥ `0.65` as a defensible "good" result — a meaningful, if modest, improvement over the 0.500 baseline — and anything above ~`0.75` as a strong result worth flagging for further investigation. Scores near `0.50–0.55` would indicate the available features carry little useful signal over the naive baseline.

Because this is decision support, ROC-AUC will be reported alongside the declining base rate (`0.542`), the constant-probability baseline (`0.500`), and a simple baseline (e.g., the pipeline's existing rule-based queue), not as proof of causation or of predicting future search-engine behavior.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import roc_auc_score

baseline_probability = df["is_declining_label"].mean()
baseline_auc = roc_auc_score(
    df["is_declining_label"],
    [baseline_probability] * len(df),
)

print("Declining base rate:", round(baseline_probability, 3))
print("Constant-probability baseline ROC-AUC:", round(baseline_auc, 3))


Declining base rate: 0.542
Constant-probability baseline ROC-AUC: 0.5


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = **one content item (page)**, identified by `content_id`.

Verified: 30,000 rows, 30,000 unique `content_id` values → one row per content item, no duplicates.

The slice below shows this: each row carries that page's identifier, its client (`client_id`), the label and the columns it's derived from, and its search/engagement/content features.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

analysis_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "content_type",
]

analysis_slice = df[analysis_columns].copy()

print("Rows:", len(analysis_slice))
print("Unique content items:", analysis_slice["content_id"].nunique())
print("One row per content item:", len(analysis_slice) == analysis_slice["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Rows per client (median):", df.groupby("client_id").size().median())
display(analysis_slice.head())

Rows: 30000
Unique content items: 30000
One row per content item: True
Unique clients: 32
Rows per client (median): 567.0


,content_id,client_id,trend_direction,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,content_type
0,content_304f48230142,client_f369cb89fc,down,1,3803,29,0.76,10.6,187,20,keyword article
1,content_a1fb4e703a9e,client_4e07408562,down,1,15320,7,0.05,20.3,445,25,keyword article
2,content_9aa793d4d895,client_7f2253d7e2,down,1,12581,11,0.09,36.5,141,20,keyword article
3,content_331d6c4de07b,client_19581e27de,stable,0,11751,58,0.49,6.2,463,22,keyword article
4,content_d99b7a2d90ca,client_3fdba35f04,down,1,19140,24,0.13,44.0,263,14,keyword article


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as "review every page older than 365 days" is useful as a baseline but too simple here. The candidate set has 13 non-leaky features (visibility, engagement, freshness, structure, search context), and earlier exploration already showed single signals like content age are weak on their own  older pages were not reliably declining. The real pattern likely depends on combinations of signals, which may interact nonlinearly and vary across the 32 clients in the dataset  too messy for one threshold to capture.

ML is justified only if it beats the fixed rule during honest validation. The result stays decision support: editors and SEO specialists review the ranked pages and decide what to do.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "search_volume",
    "competition",
    "content_type",
    "main_intent",
]

leakage_columns = {"trend_direction", "trend_pct", "is_declining_label"}
assert not leakage_columns.intersection(candidate_features)

print("Candidate features:", len(candidate_features))
print("Leakage columns excluded:", sorted(leakage_columns))
print("ML will be compared with a simple baseline before making any claim.")

Candidate features: 13
Leakage columns excluded: ['is_declining_label', 'trend_direction', 'trend_pct']
ML will be compared with a simple baseline before making any claim.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.